In [6]:
"""
02_extract_flood.py
Extracts daily simulated river discharge (m^3/s) from Copernicus GloFAS via Open-Meteo
across Western Suburbs hydrological catchments (Mithi, Oshiwara, Poisar, Dahisar).
"""

'\n02_extract_flood.py\nExtracts daily simulated river discharge (m^3/s) from Copernicus GloFAS via Open-Meteo\nacross Western Suburbs hydrological catchments (Mithi, Oshiwara, Poisar, Dahisar).\n'

In [7]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
import pandas as pd
import requests

In [10]:
# Strategic coordinate anchors near Mumbai Western drainage basins
FLOOD_ZONES = [
    {"zone_id": 0, "zone_name": "Bandra-Santacruz", "basin": "Mithi River Basin", "lat": 19.0600, "lon": 72.8500},
    {"zone_id": 1, "zone_name": "Andheri-VileParle", "basin": "Oshiwara Basin", "lat": 19.1200, "lon": 72.8400},
    {"zone_id": 2, "zone_name": "Goregaon-Malad", "basin": "Poisar Basin", "lat": 19.1700, "lon": 72.8500},
    {"zone_id": 3, "zone_name": "Kandivali-Borivali", "basin": "Dahisar River Basin", "lat": 19.2300, "lon": 72.8600},
]

def fetch_flood_discharge(start_date="2023-01-01", end_date="2024-12-31"):
    print("Fetching GloFAS hydrological river discharge from Open-Meteo...")
    
    latitudes = ",".join(str(z["lat"]) for z in FLOOD_ZONES)
    longitudes = ",".join(str(z["lon"]) for z in FLOOD_ZONES)

    api_url = "https://flood-api.open-meteo.com/v1/flood"
    params = {
        "latitude": latitudes,
        "longitude": longitudes,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "river_discharge",
        "models": "seamless_v4"
    }

    response = requests.get(api_url, params=params)
    response.raise_for_status()
    data = response.json()

    # When querying multiple coordinates, Open-Meteo returns a list of dictionaries
    if isinstance(data, dict):
        data = [data]

    all_zone_dfs = []
    for idx, location in enumerate(data):
        meta = FLOOD_ZONES[idx]
        daily = location.get("daily", {})
        
        df_zone = pd.DataFrame({
            "date": pd.to_datetime(daily.get("time", [])).date,
            "zone_id": meta["zone_id"],
            "zone_name": meta["zone_name"],
            "basin_name": meta["basin"],
            "river_discharge_m3s": daily.get("river_discharge", [])
        })
        all_zone_dfs.append(df_zone)

    df_flood = pd.concat(all_zone_dfs, ignore_index=True)

    # Calculate 3-day rolling discharge anomaly to detect saturated soil runoff
    df_flood["discharge_3d_rolling_avg"] = df_flood.groupby("zone_name")["river_discharge_m3s"].transform(
        lambda s: s.rolling(window=3, min_periods=1).mean()
    )

    # Save to disk
    os.makedirs("backend/data/processed", exist_ok=True)
    full_output = "backend/data/processed/mumbai_flood_discharge.csv"
    sample_output = "backend/data/processed/flood_sample.csv"

    df_flood.to_csv(full_output, index=False)
    df_flood.head(10).to_csv(sample_output, index=False)

    print(f"Extracted {len(df_flood)} daily discharge records.")
    print(f"Full CSV saved to:   {full_output} (Ignored by Git)")
    print(f"Sample CSV saved to: {sample_output} (Committed to Git)")

if __name__ == "__main__":
    fetch_flood_discharge()

Fetching GloFAS hydrological river discharge from Open-Meteo...
Extracted 2924 daily discharge records.
Full CSV saved to:   backend/data/processed/mumbai_flood_discharge.csv (Ignored by Git)
Sample CSV saved to: backend/data/processed/flood_sample.csv (Committed to Git)
